# 韩国关税署 2024-09 复用评价段官网索引一次传输修复

仅补一个零响应月；保留原45月来源记录，44行逐字节语义不变。无附件、SOXX结果、拟合或账户。

In [1]:
import hashlib,json,os,re,subprocess,tempfile,urllib.request,urllib.error,urllib.parse
from datetime import datetime,timezone,date
from pathlib import Path
from bs4 import BeautifulSoup
import pandas as pd,qlib
from qlib.workflow import R
from src.data.nas_market_data import publish_nas_snapshot,materialize_nas_snapshot,archive_nas_recorder,materialize_nas_evidence,_settings
root=Path.cwd();card=root/'research/kcs_early20_2024_sep_held_index_transport_repair_20261023';plan=json.loads((card/'plan.json').read_text());sha=lambda p:hashlib.sha256(Path(p).read_bytes()).hexdigest();assert plan['frozen'] and not (card/'run_state.json').exists()
source=root/plan['parent']['card'];ss=json.loads((source/'summary.json').read_text());assert sha(source/'summary.json')==plan['parent']['summary_sha256'] and sha(source/'result.json')==plan['parent']['result_sha256'] and ss['NAS_readback']['verified']
source_artifact=source/'qlib_recorder'/ss['experiment_id']/ss['recorder_id']/'artifacts/analysis/monthly_index';assert sha(source_artifact)==plan['parent']['monthly_index_sha256']
raw=root/'data/snapshots/kcs_early20_2024_sep_held_index_transport_repair_20261023';raw.mkdir(parents=True,exist_ok=True);assert not any(raw.iterdir())
state={'status':'started','plan_sha256':sha(card/'plan.json'),'GET_attempts':1,'source_Recorder_loads':0,'SOXX_outcome_reads':0,'model_fits':0,'native_accounts':0};(card/'run_state.json').write_text(json.dumps(state,indent=2)+'\n')
url=plan['exact_GET'];t=plan['transport'];started=datetime.now(timezone.utc).isoformat();body=b'';headers={};status=None;error=None;final_url=url
try:
 req=urllib.request.Request(url,headers={'User-Agent':t['user_agent']});res=urllib.request.urlopen(req,timeout=t['total_timeout_seconds']);status=res.status;final_url=res.url;headers=dict(res.headers.items());body=res.read(t['max_response_bytes']+1);res.close()
except urllib.error.HTTPError as exc:status=exc.code;final_url=exc.url;headers=dict(exc.headers.items());body=exc.read(t['max_response_bytes']+1);error='HTTPError'
except Exception as exc:error=type(exc).__name__+':'+str(exc)[:180]
ended=datetime.now(timezone.utc).isoformat();page=raw/'2024-09.response';page.write_bytes(body);head=raw/'2024-09.headers.json';head.write_text(json.dumps(headers,indent=2)+'\n')
receipt={'url':url,'final_url':final_url,'started_utc':started,'ended_utc':ended,'http_status':status,'content_type':headers.get('Content-Type',headers.get('content-type','')),'body_sha256':sha(page),'body_bytes':len(body),'headers_sha256':sha(head),'error':error};(raw/'capture_receipt.json').write_text(json.dumps(receipt,indent=2)+'\n')
accepted=[];candidates=[];reason=None
if status==200 and len(body)<=t['max_response_bytes'] and 'html' in receipt['content_type'].lower():
 soup=BeautifulSoup(body,'html.parser')
 for a in soup.find_all('a'):
  href=a.get('href') or ''
  if 'pressReleaseView.do' not in href:continue
  title=a.get_text(' ',strip=True);compact=re.sub(r'\s+','',title);id=(urllib.parse.parse_qs(urllib.parse.urlsplit(href).query).get('newsId') or [None])[0]
  if not id:continue
  info={'newsId':id,'title_excerpt':title[:240],'href':urllib.parse.urljoin(url,href)};candidates.append(info)
  if all(z in compact for z in ('9월1일','20일','수출입현황','관세청')):accepted.append(info)
 reason='unique_match' if len(accepted)==1 else 'missing_or_ambiguous'
else:reason='transport_or_format_failure'
manifest={'schema_version':1,'experiment':plan['experiment'],'plan_sha256':state['plan_sha256'],'capture_utc':datetime.now(timezone.utc).isoformat(),'receipt':receipt,'candidates':candidates,'accepted':accepted,'reason':reason,'historical_first_public_bytes_qualified':False};(raw/'source_manifest.json').write_text(json.dumps(manifest,ensure_ascii=False,indent=2)+'\n');(card/'source_receipt.json').write_text(json.dumps(manifest,ensure_ascii=False,indent=2)+'\n')
files={str(p.relative_to(root)):'raw' for p in raw.iterdir() if p.is_file()}
try:
 binding=publish_nas_snapshot(workspace_root=root,source_manifest=raw/'source_manifest.json',files=files,as_of=date.today().isoformat(),source_metadata={'experiment':plan['experiment'],'source':'KCS 2024-09 held official list repair'})
 (card/'source_nas_binding.json').write_text(json.dumps(binding,indent=2)+'\n')
 with tempfile.TemporaryDirectory(prefix='kcs-2024sep-cold-') as temp:
  out=materialize_nas_snapshot(binding,workspace_root=Path(temp))
  for obj in binding['objects']:assert sha(Path(temp)/obj['local_path'])==obj['sha256']
 assert binding['verified'] and out['restored_objects']==len(files);nas_ok=True
except Exception as exc:nas_ok=False;binding=None;manifest['NAS_failure']=type(exc).__name__+':'+str(exc)[:180]
result={'decision':'official_KCS_2024Sep_held_index_repaired_source_only' if nas_ok and len(accepted)==1 else 'needs_KCS_2024Sep_held_index_or_NAS','GET_attempts':1,'receipt':receipt,'reason':reason,'accepted':accepted,'source_NAS_id':binding['source_id'] if binding else None,'source_NAS_readback_verified':nas_ok,'SOXX_outcome_reads':0,'model_fits':0,'native_accounts':0,'historical_first_public_bytes_qualified':False}
if nas_ok and len(accepted)==1:
 os.environ['MLFLOW_ALLOW_FILE_STORE']='true';qlib.init(provider_uri=str(root/'data/qlib/minute_information_pit_allocator_20261012/daily_prices'),region='us',kernels=1)
 with R.uri_context(str(source/'qlib_recorder')):
  original=R.get_recorder(recorder_id=ss['recorder_id'],experiment_id=ss['experiment_id']).load_object('analysis/monthly_index')
 state['source_Recorder_loads']=1;(card/'run_state.json').write_text(json.dumps(state,indent=2)+'\n')
 assert len(original)==45 and original.month.is_unique and original.loc[original.month.eq('2024-09'),'parse_status'].iloc[0]=='transport_or_format_unknown'
 repaired=original.copy();mask=repaired.month.eq('2024-09');repaired.loc[mask,'http_status']=200;repaired.loc[mask,'parse_status']='one_official_list_candidate';repaired.loc[mask,'candidate_count']=len(candidates);repaired.loc[mask,'accepted_count']=1;repaired.loc[mask,'newsId']=accepted[0]['newsId'];repaired.loc[mask,'source_url']=accepted[0]['href']
 pd.testing.assert_frame_equal(repaired.loc[~mask],original.loc[~mask],check_exact=True)
 assert len(repaired)==45 and repaired.accepted_count.eq(1).sum()==45
 with R.start(experiment_name=plan['experiment'],uri=str(card/'qlib_recorder')):
  rec=R.get_recorder();rec.save_objects(artifact_path='inputs',plan=plan,parent_summary=ss);rec.save_objects(artifact_path='analysis',repaired_index=repaired,repair_receipt=manifest,result=result);rec.log_metrics(held_months=45,unique_pages=45,new_model_fits=0,new_native_accounts=0);rid,eid=rec.id,str(rec.experiment_id)
 settings=_settings(root);rb=card/'nas_binding.json';rb.write_text(json.dumps({'schema_version':'recorder_evidence_binding_v1','authority':'yanxuan_nas','source_id':plan['experiment'],'ssh':settings['ssh'],'source_plan':'plan.json'},indent=2)+'\n')
 evidence=archive_nas_recorder(rb,workspace_root=root,card=card,recorder_id=rid,experiment_id=eid)
 with tempfile.TemporaryDirectory(prefix='kcs-2024sep-rec-cold-') as temp:
  materialize_nas_evidence(rb,card=Path(temp));rel=Path('qlib_recorder')/eid/rid/'artifacts/analysis/repaired_index';assert sha(card/rel)==sha(Path(temp)/rel)
 result.update(repaired_months=45,recorder_id=rid,experiment_id=eid,Recorder_NAS_sha256=evidence['sha256'],Recorder_NAS_readback_verified=True,unchanged_44_rows=True)
state.update(status='completed' if result['decision'].startswith('official_') else 'stopped',reason=reason);(card/'run_state.json').write_text(json.dumps(state,indent=2)+'\n');(card/'result.json').write_text(json.dumps(result,ensure_ascii=False,indent=2)+'\n');print(json.dumps({k:result.get(k) for k in ('decision','reason','accepted','repaired_months','source_NAS_readback_verified','Recorder_NAS_readback_verified')},ensure_ascii=False))


{"decision": "official_KCS_2024Sep_held_index_repaired_source_only", "reason": "unique_match", "accepted": [{"newsId": "156651567", "title_excerpt": "’24년 9월 1일 ~ 9월 20일 수출입 현황 ’24  년  9  월  1  일  ~ 9  월  20  일 수출입 현황  <  요 약  >  관세청은  23  일  ,  9  월  1  일  ~20  일 기간의 수출입 현황을 발표  했다  .  수출은  356  억 달러  로 전년동기대비  △  1.1%  감소  ,  수입은  348  억 달러  로  ... 2024-09-23 관세청", "href": "https://www.korea.kr/briefing/pressReleaseView.do?newsId=156651567&pageIndex=1&repCodeType=&repCode=B00003&startDate=2024-09-20&endDate=2024-09-27&srchWord=수출입&period="}], "repaired_months": 45, "source_NAS_readback_verified": true, "Recorder_NAS_readback_verified": true}


# 2024 年 9 月韩国关税署前 20 日原稿索引修复

原冻结的 2023-01 至 2026-09 **45 月**复用评价段官方索引，仅 **2024-09** 因 TLS 握手超时而未知。本卡依新的单月固定计划，对完全相同官网列表 URL **只重取一次**：HTTP 200，得到唯一的 2024-09-23 关税署《9 月 1–20 日进出口现况》页面 `newsId=156651567`。原始列表字节、头、UTC 时钟与 SHA-256 已经 Yanxuan NAS 远端校验、限定恢复。

原生 Qlib Recorder 只替换这一个未知月的来源 ID、URL 和状态，**其余 44 行逐字段不变**；新索引 **45/45 月各有唯一官方发布页**，新 Recorder 也已 NAS 定向恢复。原 44/45 Recorder 与握手超时收据保留，不能把机械修复说成新市场数据或策略前瞻日。

结论 `official_KCS_2024Sep_held_index_repaired_source_only`：评价段官网**页面身份**完整，附件原始版本、半导体同比、首次公开时刻、修订和美股决策时点仍待分别核验。零 SOXX 结果、拟合、账户或 SOXX 超额结论。官方稿件页：[2024-09-23 关税署原稿](https://www.korea.kr/briefing/pressReleaseView.do?newsId=156651567)。
